In [ ]:
import numpy as np
import re
from Bio.SeqUtils import IsoelectricPoint as IP

AA = "ACDEFGHIKLMNPQRSTVWY"
KD = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8, "G": -0.4, "H": -3.2, "I": 4.5,
    "K": -3.9, "L": 3.8, "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5, "S": -0.8,
    "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3
}
HYDROPHOBIC = set(list("AILMFWVY"))
POS = set(list("KRH"))
NEG = set(list("DE"))
POLAR = set(list("STNQ"))
AROMATIC = set(list("FWY"))

def aac(seq: str):
    L = len(seq)
    counts = {a: 0 for a in AA}
    for ch in seq:
        if ch in counts:
            counts[ch] += 1
    return np.array([counts[a]/L for a in AA], dtype=float)

def frac_set(seq: str, aset: set):
    L = len(seq)
    return sum(1 for ch in seq if ch in aset) / L

def mean_kd(seq: str):
    return float(np.mean([KD.get(ch, 0.0) for ch in seq]))

def max_kd_window(seq: str, win: int = 19):
    vals = np.array([KD.get(ch, 0.0) for ch in seq], dtype=float)
    if len(vals) < win:
        return float(np.mean(vals))
    csum = np.cumsum(np.insert(vals, 0, 0.0))
    window_means = (csum[win:] - csum[:-win]) / win
    return float(window_means.max())

def shannon_entropy_aac(seq: str):
    p = aac(seq)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def basic_cluster_score(seq: str, region: int = 50):
    s = seq[:region]
    return len(re.findall(r"[KR]{3,}", s))

def signal_peptide_proxy(seq: str, n: int = 25):
    s = seq[:n]
    hydro_frac = frac_set(s, HYDROPHOBIC)
    kd_mean = float(np.mean([KD.get(ch, 0.0) for ch in s])) if len(s) else 0.0
    motif = 1 if re.search(r"A.A", s[10:25] if len(s) >= 25 else s) else 0
    return hydro_frac, kd_mean, float(motif)

def featurize(seq: str):
    seq = seq.upper().replace("*", "")
    L = len(seq)

    # pI
    try:
        pi = float(IP.IsoelectricPoint(seq).pi())
    except Exception:
        pi = np.nan

    sp_hydro, sp_kd, sp_motif = signal_peptide_proxy(seq, 25)

    feats = np.concatenate([
        aac(seq),
        np.array([
            L,
            mean_kd(seq),
            max_kd_window(seq, 19),
            frac_set(seq, HYDROPHOBIC),
            frac_set(seq, POLAR),
            frac_set(seq, POS),
            frac_set(seq, NEG),
            frac_set(seq, AROMATIC),
            shannon_entropy_aac(seq),
            pi,
            basic_cluster_score(seq, 50),
            sp_hydro,
            sp_kd,
            sp_motif
        ], dtype=float)
    ])
    return feats

feat_names = [f"aac_{a}" for a in AA] + [
    "len","mean_kd","max_kd_win19","frac_hydrophobic","frac_polar","frac_pos","frac_neg",
    "frac_aromatic","entropy_aac","pI","kr_cluster_50","sp_hydro_frac_25","sp_kd_mean_25","sp_motif_AxA"
]

def build_Xy(df):
    X = np.vstack([featurize(s) for s in df["sequence"].tolist()])
    y = df["y"].astype(int).values
    return X, y

X_train, y_train = build_Xy(train_df)
X_test, y_test = build_Xy(test_df)

print(X_train.shape, X_test.shape)